# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

# ---------------- LABEL (built here, never used as a feature) ----------------
y = (df["trend_direction"] == "down").astype(int)
print(f"Label positive rate: {y.mean():.3f}")

# ---------------- NUMERIC BASE ----------------
NUMERIC = [c for c in [
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "word_count", "char_count",
    "avg_position", "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
] if c in df.columns]

X = df[NUMERIC].copy()

# avg_position == 0 means "no position data", NOT rank zero
if "avg_position" in X.columns:
    X["has_position_data"] = (X["avg_position"] > 0).astype(int)
    X["avg_position"] = X["avg_position"].replace(0, np.nan)

# ---------------- ENGINEERED ----------------
eps = 1e-9
if {"clicks_90d", "impressions_90d"}.issubset(X.columns):
    X["clicks_per_impression"] = X["clicks_90d"] / (X["impressions_90d"] + eps)
if {"sessions_90d", "impressions_90d"}.issubset(X.columns):
    X["sessions_per_impression"] = X["sessions_90d"] / (X["impressions_90d"] + eps)
if {"days_since_last_update", "content_age_days"}.issubset(X.columns):
    X["staleness_ratio"] = X["days_since_last_update"] / (X["content_age_days"] + eps)
for c in ["impressions_90d", "sessions_90d", "search_volume", "word_count"]:
    if c in X.columns:
        X[f"log_{c}"] = np.log1p(X[c].clip(lower=0))

# ---------------- MISSINGNESS FLAGS (before filling) ----------------
for c in list(X.columns):
    if X[c].isna().any():
        X[f"has_{c}"] = X[c].notna().astype(int)

# ---------------- CATEGORICAL ----------------
CATEGORICAL = [c for c in ["content_type", "main_intent", "competition_level",
                           "age_tier", "freshness_tier", "word_count_tier",
                           "impression_tier", "position_tier"] if c in df.columns]

cats = df[CATEGORICAL].astype("object").fillna("__missing__")
cat_dummies = pd.get_dummies(cats, prefix=CATEGORICAL, dummy_na=False).astype(int)

# ---------------- ASSEMBLE ----------------
X = pd.concat([X, cat_dummies], axis=1)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

groups = df["client_id"]   # grouping key only -- never a feature

print(f"\nFeature vector: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"  numeric + engineered: {X.shape[1] - cat_dummies.shape[1]}")
print(f"  one-hot categorical:  {cat_dummies.shape[1]}")
print(f"  categoricals encoded: {CATEGORICAL}")
X.head(3)

Loaded 30,000 rows x 44 columns
Label positive rate: 0.542

Feature vector: 30,000 rows x 69 features
  numeric + engineered: 35
  one-hot categorical:  34
  categoricals encoded: ['content_type', 'main_intent', 'competition_level', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,days_with_impressions,days_with_sessions,content_age_days,days_since_last_update,word_count,char_count,...,word_count_tier___missing__,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,3803,29,17,0,88,13,187,20,3221.0,20457.0,...,0,0,1,0,0,0,0,0,1,0
1,15320,7,9,0,88,9,445,25,2481.0,15562.0,...,0,0,1,0,0,0,0,1,0,0
2,12581,11,11,0,88,11,141,20,3515.0,23643.0,...,0,0,1,0,0,0,0,1,0,0


The feature vector is built from observable signals only — measurements of what happened,
never a decision someone already made. Four things happen here:

1. **Numeric features** are taken as-is, with two corrections. `avg_position == 0` means
   "no data", not rank zero, so it becomes missing and gets a `has_position_data` flag.
   Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are already
   ×100 percentages, so they are not rescaled.
2. **Engineered features** are ratios and logs built only from the feature window —
   clicks per impression, sessions per impression, a staleness-to-age ratio, and log
   transforms for the heavily skewed volume columns.
3. **Categorical features** are one-hot encoded, with a missing value kept as its own
   category rather than silently dropped.
4. **Missing values** are filled with 0, but every filled column gets a `has_*` indicator
   alongside it. This matters more than it sounds: missingness in this dataset tracks
   `content_type` — one type is almost entirely missing keyword data, another is heavily
   missing word count. A blind `fillna(0)` would therefore smuggle a content-type signal
   into the model through the back door. The flags make that signal explicit instead of hidden.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evidence for the table above: missingness, and that it tracks content_type
print("=== Missingness by column (top 10) ===")
miss = df[NUMERIC + CATEGORICAL].isna().mean().sort_values(ascending=False)
print((miss[miss > 0].head(10) * 100).round(1).to_string(), "\n(% missing)\n")

if "content_type" in df.columns:
    print("=== Why a blind fillna(0) would be a mistake ===")
    probe = [c for c in ["search_volume", "word_count"] if c in df.columns]
    by_type = df.groupby("content_type")[probe].apply(lambda g: g.isna().mean() * 100).round(1)
    print(by_type.to_string())
    print("\nMissingness is NOT uniform across content types -> filling with 0 silently")
    print("encodes content type. The has_* flags make that explicit instead.\n")

print("=== Flags created ===")
flags = [c for c in X.columns if c.startswith("has_")]
print(f"{len(flags)} indicator columns: {flags}")

=== Missingness by column (top 10) ===
char_count           25.7
word_count           25.7
word_count_tier      25.7
competition_level     8.7
cpc                   8.2
competition           8.2
search_volume         8.2
main_intent           7.9
scroll_rate           0.4 
(% missing)

=== Why a blind fillna(0) would be a mistake ===
                    search_volume  word_count
content_type                                 
comparison article            0.0         0.0
feedly article              100.0         0.0
keyword article               1.4        28.3

Missingness is NOT uniform across content types -> filling with 0 silently
encodes content type. The has_* flags make that explicit instead.

=== Flags created ===
10 indicator columns: ['has_position_data', 'has_word_count', 'has_char_count', 'has_avg_position', 'has_scroll_rate', 'has_search_volume', 'has_competition', 'has_cpc', 'has_log_search_volume', 'has_log_word_count']


The question that decides whether a feature is allowed is not "is it useful?" but
**"did it exist before the moment of prediction?"** Everything below is a trailing-90-day
measurement — recorded before any decision about the page was made.

| Feature group | What it means | Missing handling | Available before prediction? |
|---|---|---|---|
| `impressions_90d`, `clicks_90d`, `sessions_90d` | Search and analytics volume over the trailing window | Fill 0 + `has_*` flag | Yes — already-occurred counts |
| `ai_sessions_90d`, `ai_traffic_pct` | Sessions arriving from AI tools | Fill 0 + flag; sparse by nature | Yes — observed |
| `avg_position` | Mean search position | `0` → missing + `has_position_data` | Yes — observed ranking |
| `ctr`, `engagement_rate`, `scroll_rate` | Rates, already ×100 percentages | Fill 0 + flag; **not rescaled** | Yes — computed from past behaviour |
| `content_age_days`, `days_since_last_update` | Lifecycle timing | Fill 0 + flag | Yes — knowable at any moment |
| `word_count`, `char_count` | Content size | Fill 0 + flag | Yes — property of the page as it stands |
| `search_volume`, `competition`, `cpc` | Keyword-market context | Fill 0 + flag | Yes — external market data, not outcome data |
| `clicks_per_impression`, `sessions_per_impression` | Engineered efficiency ratios | Built after fills, so no new missingness | Yes — derived only from feature-window columns |
| `staleness_ratio` | How much of a page's life it has gone unedited | Same | Yes |
| `log_*` | Log transforms of skewed volume columns | Same | Yes — monotone transform, adds no information |
| One-hot categoricals | Content type, intent, competition level, and transparent tiers | Missing kept as its own `__missing__` category | Yes — descriptive buckets, not decisions |

**One honest caveat about the tiers.** `age_tier`, `freshness_tier`, `position_tier` and the
rest are precalculated buckets of columns already in the vector. They add interpretability,
not new information, and they are transparent derivations rather than product decisions —
which is why they are allowed. If a result depended entirely on a tier and vanished without
it, that would be worth investigating rather than celebrating.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import roc_auc_score

# --- CHECK 1 & 2 & 3: forbidden columns ---
BANNED = ["trend_direction", "trend_pct", "is_declining_label",
          "content_id", "client_id",
          "health_score", "priority_score", "action_type", "refresh_tier",
          "needs_ctr_fix", "is_quick_win"]

found = [c for c in X.columns if any(b in c.lower() for b in BANNED)]
print("=== Checks 1-3: label-derived, ID, and product-flag columns in features ===")
print("FOUND:", found if found else "NONE — clean")

present_in_source = [c for c in df.columns if any(b in c.lower() for b in
                     ["health_score", "priority_score", "action_type", "refresh_tier"])]
print("Product decision flags present in the source data at all:",
      present_in_source if present_in_source else "NONE — absent by design, as documented")

# --- CHECK 4: single-feature separation ---
print("\n=== Check 4: single-feature AUC (leak signature hunt) ===")
rows = []
for c in X.columns:
    if X[c].nunique() > 1:
        try:
            rows.append((c, roc_auc_score(y, X[c])))
        except Exception:
            pass
single = pd.DataFrame(rows, columns=["feature", "auc_alone"])
single["dist_from_random"] = (single["auc_alone"] - 0.5).abs()
top = single.sort_values("dist_from_random", ascending=False).head(8)
print(top.round(3).to_string(index=False))

worst = top["dist_from_random"].max()
print(f"\nStrongest single feature sits {worst:.3f} from random.")
print("Flag threshold: > 0.45 would indicate a leak." ,
      "-> PASS" if worst <= 0.45 else "-> INVESTIGATE")

# --- CHECK 5: positive control ---
print("\n=== Check 5: positive control — plant a known leak, confirm it's detected ===")
X_leak = X.copy()
X_leak["LEAK_trend_pct"] = df["trend_pct"].fillna(0)

gkf = GroupKFold(n_splits=3)
rf = lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clean_auc = cross_val_score(rf(), X,      y, groups=groups, cv=gkf, scoring="roc_auc").mean()
leak_auc  = cross_val_score(rf(), X_leak, y, groups=groups, cv=gkf, scoring="roc_auc").mean()

print(f"Clean feature set      — grouped CV ROC-AUC: {clean_auc:.3f}")
print(f"With trend_pct planted — grouped CV ROC-AUC: {leak_auc:.3f}")
print(f"Jump: +{leak_auc - clean_auc:.3f}  <- this is what leakage looks like here.")
print("trend_pct stays OUT. The clean number is the one I report.")

leak_single = roc_auc_score(y, df["trend_pct"].fillna(0))
print(f"\n(trend_pct alone scores AUC {leak_single:.3f} — "
      f"{'caught by check 4' if abs(leak_single-0.5) > 0.45 else 'note: below the check-4 threshold'})")

# --- PRIVACY CHECK ---
print("\n=== Privacy check: no raw identifying text anywhere ===")
RISKY = ["url", "domain", "title", "query", "keyword_text", "client_name", "page_path", "slug"]
risky_cols = [c for c in df.columns if any(r in c.lower() for r in RISKY)]
print("Columns with risky names:", risky_cols if risky_cols else "NONE")

obj_cols = df.select_dtypes(include="object").columns.tolist()
print("Text columns present:", obj_cols)
print("\nAll text columns are pseudonymous IDs or categorical buckets.")
print("No raw queries, URLs, domains, titles or client names exist in this release.")

=== Checks 1-3: label-derived, ID, and product-flag columns in features ===
FOUND: NONE — clean
Product decision flags present in the source data at all: NONE — absent by design, as documented

=== Check 4: single-feature AUC (leak signature hunt) ===
                feature  auc_alone  dist_from_random
sessions_per_impression      0.390             0.110
        staleness_ratio      0.597             0.097
       content_age_days      0.409             0.091
    log_impressions_90d      0.584             0.084
        impressions_90d      0.584             0.084
  days_with_impressions      0.579             0.079
    impression_tier_low      0.433             0.067
        age_tier_91-180      0.566             0.066

Strongest single feature sits 0.110 from random.
Flag threshold: > 0.45 would indicate a leak. -> PASS

=== Check 5: positive control — plant a known leak, confirm it's detected ===
Clean feature set      — grouped CV ROC-AUC: 0.650
With trend_pct planted — grouped CV R

Attacking my own feature set, on the assumption that something got through. Five checks:

1. **Label-derived columns.** `is_declining_label` comes from `trend_direction`, which is
   bucketed from `trend_pct`. Neither may appear as a feature — doing so would let the model
   read the answer off a reworded copy of itself.
2. **Identifier columns.** `content_id` and `client_id` are pseudonyms. A model that learns
   an ID has memorised, not generalised.
3. **Product decision flags.** `health_score`, `priority_score`, `action_type`, refresh flags.
   These are absent from this release by design, but the check runs anyway — an assumption
   that goes unverified is just a hope.
4. **Single-feature separation.** Any one feature that alone achieves near-perfect separation
   is a leak signature. A real signal is informative; a leak is decisive.
5. **Positive control.** Deliberately add `trend_pct` once, measure the jump, then remove it.
   Seeing the failure mode on purpose is more useful than trusting it is absent — if the
   check can't detect a leak I planted myself, it can't detect one I didn't.

**Privacy check, run alongside:** confirm no column holds raw queries, URLs, domains, page
titles, or client names. Pseudonymous IDs and aggregated metrics only.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final receipt: what shipped, what didn't
import json

excluded = {
    "trend_direction": "label source",
    "trend_pct": "label source, one step removed",
    "content_id": "pseudonymous row key — grouping/ID only",
    "client_id": "pseudonymous client key — reserved for grouped splitting",
    "product_decision_flags": "absent by design; would be a circular result",
    "raw_text_fields": "absent by design; privacy",
}

receipt = {
    "rows": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "n_categorical_encoded": int(cat_dummies.shape[1]),
    "n_missingness_flags": int(len([c for c in X.columns if c.startswith('has_')])),
    "label": "trend_direction == 'down' (same-window derived proxy)",
    "label_positive_rate": float(y.mean()),
    "grouping_key": "client_id",
    "leakage_checks_passed": bool(not found),
    "clean_grouped_cv_auc": float(clean_auc),
    "planted_leak_grouped_cv_auc": float(leak_auc),
    "excluded": excluded,
}

with open("work/outputs/w03_feature_vector_receipt.json", "w") as f:
    json.dump(receipt, f, indent=2)

print(json.dumps(receipt, indent=2))
print("\nWrote work/outputs/w03_feature_vector_receipt.json")

{
  "rows": 30000,
  "n_features": 69,
  "n_categorical_encoded": 34,
  "n_missingness_flags": 10,
  "label": "trend_direction == 'down' (same-window derived proxy)",
  "label_positive_rate": 0.5420666666666667,
  "grouping_key": "client_id",
  "leakage_checks_passed": true,
  "clean_grouped_cv_auc": 0.6497651444436825,
  "planted_leak_grouped_cv_auc": 0.9999957660465285,
  "excluded": {
    "trend_direction": "label source",
    "trend_pct": "label source, one step removed",
    "content_id": "pseudonymous row key \u2014 grouping/ID only",
    "client_id": "pseudonymous client key \u2014 reserved for grouped splitting",
    "product_decision_flags": "absent by design; would be a circular result",
    "raw_text_fields": "absent by design; privacy"
  }
}

Wrote work/outputs/w03_feature_vector_receipt.json


Every field refused, with the reason. This list is as much a part of the feature vector as
the columns that made it in.

| Excluded | Why |
|---|---|
| `trend_direction` | The label is defined from it. Using it as a feature hands the model the answer. |
| `trend_pct` | The percentage change `trend_direction` is bucketed from — the same leak, one step removed. Used once above as a planted control, never in the real vector. |
| `content_id` | Pseudonymous row key. A model that learns it has memorised rows, not patterns. |
| `client_id` | Pseudonymous client key. Reserved for grouped splitting — using it as a feature would let the model learn "this client declines" and defeat the grouped holdout. |
| `health_score`, `priority_score`, `action_type`, refresh flags | Product decision outputs. Absent from this release by design. Had they been present, using them would mean reproducing an existing rule and calling it a discovery — a circular result. |
| Raw queries, URLs, domains, titles | Not in the release, and would be a privacy violation to publish or reconstruct even if they were. |
| Any future-window metric | There is no forward window in this slice, so none exists to leak — but the rule is stated because the capstone's stronger label (prior 90 days → next 30 days) makes it live. |

**The rule behind all of it:** if a field would only be known *because* someone already made
the decision I am trying to predict, it leaks. The test is not whether the field helps — a
leaked field always helps, spectacularly, and teaches nothing.

**One limitation worth stating plainly.** This feature set is clean against the leaks I
checked for. It cannot be clean against leaks I did not think to check for, and the label
itself remains a same-window proxy rather than an observed future outcome — which is a
weaker starting position than the leakage audit alone can fix.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.